In [9]:
# ============================================================
# CHECK: Duplicate customer-month combinations
# ============================================================

import pandas as pd

df_usage = pd.read_csv(
    "../../data/cleaned/monthly_usage_features.csv"
)

duplicate_customer_month = df_usage.duplicated(
    subset=["customer_id", "month"]
).sum()

print("Duplicate customer-month rows:", duplicate_customer_month)
print("Total rows:", len(df_usage))

Duplicate customer-month rows: 0
Total rows: 30525


In [1]:
# STEP 7: Load cleaned data, define churn, and calculate tenure

import pandas as pd
import numpy as np
from pathlib import Path


# ---------------------------------------------------
# 1. Load cleaned datasets
# ---------------------------------------------------

clean_path = Path("../../data/cleaned")

subscriptions = pd.read_csv(
    clean_path / "subscriptions_clean.csv"
)

usage = pd.read_csv(
    clean_path / "usage_clean.csv"
)


# ---------------------------------------------------
# 2. Convert date columns
# ---------------------------------------------------

subscriptions["signup_date"] = pd.to_datetime(
    subscriptions["signup_date"]
)

subscriptions["churn_date"] = pd.to_datetime(
    subscriptions["churn_date"]
)

usage["month"] = pd.to_datetime(
    usage["month"]
)


# ---------------------------------------------------
# 3. Define analysis end date
# ---------------------------------------------------

analysis_end_date = pd.Timestamp("2025-12-31")


# ---------------------------------------------------
# 4. Create churn flag
# ---------------------------------------------------

subscriptions["churn_flag"] = (
    subscriptions["churned"]
    .map({
        "yes": 1,
        "no": 0
    })
)


# ---------------------------------------------------
# 5. Create tenure end date
# ---------------------------------------------------

# If churned:
# tenure ends on churn date
#
# If still active:
# tenure ends on 31 Dec 2025

subscriptions["tenure_end_date"] = (
    subscriptions["churn_date"]
    .fillna(analysis_end_date)
)


# ---------------------------------------------------
# 6. Calculate tenure in days
# ---------------------------------------------------

subscriptions["tenure_days"] = (
    subscriptions["tenure_end_date"]
    - subscriptions["signup_date"]
).dt.days


# ---------------------------------------------------
# 7. Calculate approximate tenure in months
# ---------------------------------------------------

subscriptions["tenure_months"] = (
    subscriptions["tenure_days"] / 30.44
).round(1)


# ---------------------------------------------------
# 8. Preview results
# ---------------------------------------------------

display(
    subscriptions[
        [
            "customer_id",
            "signup_date",
            "churned",
            "churn_date",
            "churn_flag",
            "tenure_days",
            "tenure_months"
        ]
    ].head(10)
)


# ---------------------------------------------------
# 9. Basic validation
# ---------------------------------------------------

print("Churn flag counts:")
print(subscriptions["churn_flag"].value_counts())

print("\nTenure summary:")
print(subscriptions["tenure_months"].describe())

print("\nNegative tenure records:")
print((subscriptions["tenure_days"] < 0).sum())

,customer_id,signup_date,churned,churn_date,churn_flag,tenure_days,tenure_months
0,1,2024-09-11,yes,2025-06-29,1,291,9.6
1,2,2025-08-14,no,NaT,0,139,4.6
2,3,2025-01-24,no,NaT,0,341,11.2
3,4,2024-08-15,yes,2024-08-20,1,5,0.2
4,5,2025-06-24,no,NaT,0,190,6.2
5,6,2024-08-11,yes,2025-03-01,1,202,6.6
6,7,2024-11-05,yes,2024-12-28,1,53,1.7
7,8,2025-10-20,no,NaT,0,72,2.4
8,9,2024-08-29,yes,2025-07-27,1,332,10.9
9,10,2025-01-26,no,NaT,0,339,11.1


Churn flag counts:
churn_flag
0    2432
1    1768
Name: count, dtype: int64

Tenure summary:
count    4200.000000
mean        6.858667
std         4.415342
min         0.000000
25%         3.100000
50%         6.100000
75%        10.400000
max        18.000000
Name: tenure_months, dtype: float64

Negative tenure records:
0


In [2]:
# STEP 8: Create tenure bands and first-month engagement features

# ---------------------------------------------------
# 1. Create tenure bands
# ---------------------------------------------------

subscriptions["tenure_band"] = pd.cut(
    subscriptions["tenure_months"],
    bins=[-1, 1, 3, 6, 12, 18, 100],
    labels=[
        "0-1 months",
        "2-3 months",
        "4-6 months",
        "7-12 months",
        "13-18 months",
        "18+ months"
    ]
)


# ---------------------------------------------------
# 2. Create signup month
# ---------------------------------------------------

subscriptions["signup_month"] = (
    subscriptions["signup_date"]
    .dt.to_period("M")
    .dt.to_timestamp()
)


# ---------------------------------------------------
# 3. Merge signup month into usage data
# ---------------------------------------------------

usage = usage.merge(
    subscriptions[
        [
            "customer_id",
            "signup_month"
        ]
    ],
    on="customer_id",
    how="left"
)


# ---------------------------------------------------
# 4. Identify first-month usage records
# ---------------------------------------------------

usage["is_first_month"] = (
    usage["month"] == usage["signup_month"]
).astype(int)


# ---------------------------------------------------
# 5. Extract first-month activity
# ---------------------------------------------------

first_month_usage = (
    usage[usage["is_first_month"] == 1]
    [
        [
            "customer_id",
            "workouts_completed",
            "minutes_active",
            "classes_booked",
            "support_tickets"
        ]
    ]
    .copy()
)


# Rename columns so they are easy to understand later

first_month_usage = first_month_usage.rename(
    columns={
        "workouts_completed": "first_month_workouts",
        "minutes_active": "first_month_minutes",
        "classes_booked": "first_month_classes",
        "support_tickets": "first_month_support_tickets"
    }
)


# ---------------------------------------------------
# 6. Merge first-month usage into subscriptions
# ---------------------------------------------------

subscriptions = subscriptions.merge(
    first_month_usage,
    on="customer_id",
    how="left"
)


# ---------------------------------------------------
# 7. Create simple first-month engagement band
# ---------------------------------------------------

subscriptions["first_month_engagement_band"] = pd.cut(
    subscriptions["first_month_workouts"],
    bins=[-1, 1, 3, 7, 1000],
    labels=[
        "Very Low",
        "Low",
        "Medium",
        "High"
    ]
)


# ---------------------------------------------------
# 8. Preview results
# ---------------------------------------------------

display(
    subscriptions[
        [
            "customer_id",
            "tenure_months",
            "tenure_band",
            "first_month_workouts",
            "first_month_minutes",
            "first_month_classes",
            "first_month_support_tickets",
            "first_month_engagement_band"
        ]
    ].head(10)
)


# ---------------------------------------------------
# 9. Validation
# ---------------------------------------------------

print("Tenure band counts:")
print(subscriptions["tenure_band"].value_counts(dropna=False))

print("\nFirst-month engagement counts:")
print(
    subscriptions[
        "first_month_engagement_band"
    ].value_counts(dropna=False)
)

,customer_id,tenure_months,tenure_band,first_month_workouts,first_month_minutes,first_month_classes,first_month_support_tickets,first_month_engagement_band
0,1,9.6,7-12 months,5,140.0,0,0,Medium
1,2,4.6,4-6 months,2,58.0,0,0,Low
2,3,11.2,7-12 months,0,0.0,0,0,Very Low
3,4,0.2,0-1 months,2,80.0,0,1,Low
4,5,6.2,7-12 months,3,79.0,0,0,Low
5,6,6.6,7-12 months,4,86.0,1,0,Medium
6,7,1.7,2-3 months,1,28.0,0,0,Very Low
7,8,2.4,2-3 months,0,0.0,0,0,Very Low
8,9,10.9,7-12 months,1,38.0,0,0,Very Low
9,10,11.1,7-12 months,4,110.0,0,0,Medium


Tenure band counts:
tenure_band
7-12 months     1606
4-6 months      1030
2-3 months       752
13-18 months     533
0-1 months       279
18+ months         0
Name: count, dtype: int64

First-month engagement counts:
first_month_engagement_band
Medium      1283
Low         1262
Very Low    1067
High         588
Name: count, dtype: int64


In [3]:
# STEP 9: Create previous-month activity features

# ---------------------------------------------------
# 1. Sort usage data correctly
# ---------------------------------------------------

usage = usage.sort_values(
    by=["customer_id", "month"]
).reset_index(drop=True)


# ---------------------------------------------------
# 2. Create previous-month activity columns
# ---------------------------------------------------

usage["prev_month_workouts"] = (
    usage.groupby("customer_id")["workouts_completed"]
    .shift(1)
)

usage["prev_month_minutes"] = (
    usage.groupby("customer_id")["minutes_active"]
    .shift(1)
)

usage["prev_month_classes"] = (
    usage.groupby("customer_id")["classes_booked"]
    .shift(1)
)

usage["prev_month_support_tickets"] = (
    usage.groupby("customer_id")["support_tickets"]
    .shift(1)
)


# ---------------------------------------------------
# 3. Create previous-month workout engagement band
# ---------------------------------------------------

usage["prev_month_engagement_band"] = pd.cut(
    usage["prev_month_workouts"],
    bins=[-1, 1, 3, 7, float("inf")],
    labels=[
        "Very Low",
        "Low",
        "Medium",
        "High"
    ]
)


# ---------------------------------------------------
# 4. Create support-contact flag
# ---------------------------------------------------

usage["support_contact_flag"] = (
    usage["support_tickets"] > 0
).astype(int)


# ---------------------------------------------------
# 5. Previous-month support-contact flag
# ---------------------------------------------------

usage["prev_month_support_flag"] = (
    usage["prev_month_support_tickets"] > 0
).astype(int)


# ---------------------------------------------------
# 6. Preview lag features
# ---------------------------------------------------

display(
    usage[
        [
            "customer_id",
            "month",
            "workouts_completed",
            "prev_month_workouts",
            "minutes_active",
            "prev_month_minutes",
            "support_tickets",
            "prev_month_support_tickets",
            "prev_month_engagement_band"
        ]
    ].head(20)
)


# ---------------------------------------------------
# 7. Validation
# ---------------------------------------------------

print("Previous-month engagement distribution:")

print(
    usage["prev_month_engagement_band"]
    .value_counts(dropna=False)
)

print("\nRows without previous-month activity:")

print(
    usage["prev_month_workouts"]
    .isna()
    .sum()
)

,customer_id,month,workouts_completed,prev_month_workouts,minutes_active,prev_month_minutes,support_tickets,prev_month_support_tickets,prev_month_engagement_band
0,1,2024-09-01,5,NaN,140.0,NaN,0,NaN,NaN
1,1,2024-10-01,4,5.0,116.0,140.0,0,0.0,Medium
2,1,2024-11-01,7,4.0,208.0,116.0,0,0.0,Medium
3,1,2024-12-01,5,7.0,109.0,208.0,0,0.0,Medium
4,1,2025-01-01,1,5.0,41.0,109.0,0,0.0,Medium
5,1,2025-02-01,6,1.0,119.0,41.0,0,0.0,Very Low
6,1,2025-03-01,5,6.0,167.0,119.0,3,0.0,Medium
7,1,2025-04-01,3,5.0,78.0,167.0,2,3.0,Medium
8,1,2025-05-01,7,3.0,245.0,78.0,0,2.0,Low
9,1,2025-06-01,3,7.0,111.0,245.0,1,0.0,Medium


Previous-month engagement distribution:
prev_month_engagement_band
Medium      8470
Low         6810
Very Low    6077
High        4968
NaN         4200
Name: count, dtype: int64

Rows without previous-month activity:
4200


In [4]:
# STEP 10: Create churn-month flags and monthly churn rate

# ---------------------------------------------------
# 1. Create churn month in subscriptions
# ---------------------------------------------------

subscriptions["churn_month"] = (
    subscriptions["churn_date"]
    .dt.to_period("M")
    .dt.to_timestamp()
)


# ---------------------------------------------------
# 2. Add churn information to monthly usage data
# ---------------------------------------------------

usage = usage.merge(
    subscriptions[
        [
            "customer_id",
            "churn_flag",
            "churn_month"
        ]
    ],
    on="customer_id",
    how="left"
)


# ---------------------------------------------------
# 3. Create churn-month flag
# ---------------------------------------------------

# 1 = customer cancelled during this month
# 0 = customer did not cancel during this month

usage["churn_month_flag"] = (
    (usage["month"] == usage["churn_month"]) &
    (usage["churn_flag"] == 1)
).astype(int)


# ---------------------------------------------------
# 4. Build monthly churn summary
# ---------------------------------------------------

monthly_churn = (
    usage.groupby("month")
    .agg(
        active_subscribers=("customer_id", "nunique"),
        churned_subscribers=("churn_month_flag", "sum")
    )
    .reset_index()
)


# ---------------------------------------------------
# 5. Calculate monthly churn rate
# ---------------------------------------------------

monthly_churn["monthly_churn_rate"] = (
    monthly_churn["churned_subscribers"]
    / monthly_churn["active_subscribers"]
)


# Percentage version for easier interpretation
monthly_churn["monthly_churn_rate_pct"] = (
    monthly_churn["monthly_churn_rate"] * 100
).round(2)


# ---------------------------------------------------
# 6. Preview monthly churn table
# ---------------------------------------------------

display(monthly_churn)


# ---------------------------------------------------
# 7. Basic validation
# ---------------------------------------------------

print("Total monthly cancellations counted:")
print(monthly_churn["churned_subscribers"].sum())

print("\nTotal churned customers:")
print(subscriptions["churn_flag"].sum())

print("\nAverage monthly churn rate:")
print(
    round(
        monthly_churn["monthly_churn_rate_pct"].mean(),
        2
    ),
    "%"
)

,month,active_subscribers,churned_subscribers,monthly_churn_rate,monthly_churn_rate_pct
0,2024-07-01,172,8,0.046512,4.65
1,2024-08-01,355,14,0.039437,3.94
2,2024-09-01,563,38,0.067496,6.75
3,2024-10-01,769,36,0.046814,4.68
4,2024-11-01,971,52,0.053553,5.36
5,2024-12-01,1192,54,0.045302,4.53
6,2025-01-01,1379,77,0.055838,5.58
7,2025-02-01,1546,81,0.052393,5.24
8,2025-03-01,1742,103,0.059127,5.91
9,2025-04-01,1892,98,0.051797,5.18


Total monthly cancellations counted:
1768

Total churned customers:
1768

Average monthly churn rate:
5.55 %


In [5]:
# STEP 11: Save engineered datasets for SQL and Power BI

# ---------------------------------------------------
# 1. Save customer-level engineered dataset
# ---------------------------------------------------

subscriptions.to_csv(
    "../../data/cleaned/customer_features.csv",
    index=False
)


# ---------------------------------------------------
# 2. Save monthly usage engineered dataset
# ---------------------------------------------------

usage.to_csv(
    "../../data/cleaned/monthly_usage_features.csv",
    index=False
)


# ---------------------------------------------------
# 3. Save monthly churn summary
# ---------------------------------------------------

monthly_churn.to_csv(
    "../../data/cleaned/monthly_churn_summary.csv",
    index=False
)


# ---------------------------------------------------
# 4. Confirmation
# ---------------------------------------------------

print("Saved successfully:")

print("customer_features.csv")
print("monthly_usage_features.csv")
print("monthly_churn_summary.csv")

Saved successfully:
customer_features.csv
monthly_usage_features.csv
monthly_churn_summary.csv


In [6]:
# ============================================================
# FIX: Keep only the 22 required customer_features columns
# ============================================================

import pandas as pd

# ------------------------------------------------------------
# 1. Read the current customer_features file
# ------------------------------------------------------------

file_path = "../../data/cleaned/customer_features.csv"

df = pd.read_csv(file_path)


# ------------------------------------------------------------
# 2. Check current shape and columns
# ------------------------------------------------------------

print("Current rows:", df.shape[0])
print("Current columns:", df.shape[1])

print("\nCurrent column names:")
print(df.columns.tolist())


# ------------------------------------------------------------
# 3. Define the exact columns required by PostgreSQL
# ------------------------------------------------------------

required_columns = [
    "customer_id",
    "signup_date",
    "plan_type",
    "plan_tier",
    "monthly_price",
    "acquisition_channel",
    "primary_device",
    "age_group",
    "churned",
    "churn_date",
    "cancel_reason",
    "churn_flag",
    "tenure_end_date",
    "tenure_days",
    "tenure_months",
    "tenure_band",
    "signup_month",
    "first_month_workouts",
    "first_month_minutes",
    "first_month_classes",
    "first_month_support_tickets",
    "first_month_engagement_band"
]


# ------------------------------------------------------------
# 4. Check if all required columns exist
# ------------------------------------------------------------

missing_columns = [
    col for col in required_columns
    if col not in df.columns
]

print("\nMissing required columns:")
print(missing_columns)


# ------------------------------------------------------------
# 5. Keep only required columns
# ------------------------------------------------------------

if len(missing_columns) == 0:

    df = df[required_columns].copy()

    # --------------------------------------------------------
    # 6. Overwrite the incorrect CSV with the corrected version
    # --------------------------------------------------------

    df.to_csv(
        file_path,
        index=False
    )

    print("\nCSV FIXED SUCCESSFULLY")

    print("Final rows:", df.shape[0])
    print("Final columns:", df.shape[1])

else:
    print(
        "\nDo not save yet because some required columns are missing."
    )

Current rows: 4200
Current columns: 23

Current column names:
['customer_id', 'signup_date', 'plan_type', 'plan_tier', 'monthly_price', 'acquisition_channel', 'primary_device', 'age_group', 'churned', 'churn_date', 'cancel_reason', 'churn_flag', 'tenure_end_date', 'tenure_days', 'tenure_months', 'tenure_band', 'signup_month', 'first_month_workouts', 'first_month_minutes', 'first_month_classes', 'first_month_support_tickets', 'first_month_engagement_band', 'churn_month']

Missing required columns:
[]

CSV FIXED SUCCESSFULLY
Final rows: 4200
Final columns: 22


In [7]:
# ============================================================
# CHECK: monthly_usage_features.csv before PostgreSQL import
# ============================================================

import pandas as pd

file_path = "../../data/cleaned/monthly_usage_features.csv"

df_usage = pd.read_csv(file_path)

print("Rows:", df_usage.shape[0])
print("Columns:", df_usage.shape[1])

print("\nColumn names:")
print(df_usage.columns.tolist())

Rows: 30525
Columns: 19

Column names:
['customer_id', 'month', 'workouts_completed', 'minutes_active', 'classes_booked', 'support_tickets', 'minutes_active_missing_flag', 'signup_month', 'is_first_month', 'prev_month_workouts', 'prev_month_minutes', 'prev_month_classes', 'prev_month_support_tickets', 'prev_month_engagement_band', 'support_contact_flag', 'prev_month_support_flag', 'churn_flag', 'churn_month', 'churn_month_flag']


In [10]:
print("test")

test


In [11]:
import pandas as pd

df_usage = pd.read_csv(
    "../../data/cleaned/monthly_usage_features.csv"
)

print("File loaded")
print("Total rows:", len(df_usage))
print("Total columns:", len(df_usage.columns))

File loaded
Total rows: 30525
Total columns: 19


In [12]:
duplicate_customer_month = df_usage.duplicated(
    subset=["customer_id", "month"]
).sum()

print("Duplicate customer-month rows:", duplicate_customer_month)

Duplicate customer-month rows: 0


In [13]:
# ============================================================
# STEP 5A: Verify monthly_usage_features.csv
# ============================================================

import pandas as pd

df_usage = pd.read_csv(
    "../../data/cleaned/monthly_usage_features.csv"
)

print("Rows:", df_usage.shape[0])
print("Columns:", df_usage.shape[1])

print("\nFirst 3 rows:")
display(df_usage.head(3))

Rows: 30525
Columns: 19

First 3 rows:


,customer_id,month,workouts_completed,minutes_active,classes_booked,support_tickets,minutes_active_missing_flag,signup_month,is_first_month,prev_month_workouts,prev_month_minutes,prev_month_classes,prev_month_support_tickets,prev_month_engagement_band,support_contact_flag,prev_month_support_flag,churn_flag,churn_month,churn_month_flag
0,1,2024-09-01,5,140.0,0,0,0,2024-09-01,1,NaN,NaN,NaN,NaN,NaN,0,0,1,2025-06-01,0
1,1,2024-10-01,4,116.0,0,0,0,2024-09-01,0,5.0,140.0,0.0,0.0,Medium,0,0,1,2025-06-01,0
2,1,2024-11-01,7,208.0,0,0,0,2024-09-01,0,4.0,116.0,0.0,0.0,Medium,0,0,1,2025-06-01,0


In [14]:
# ============================================================
# CHECK THE ACTUAL MONTHLY USAGE CSV
# ============================================================

import pandas as pd

df = pd.read_csv(
    "../../data/cleaned/monthly_usage_features.csv"
)

print("Rows:", len(df))
print("Columns:", len(df.columns))

print("\nColumn names:")
print(df.columns.tolist())

Rows: 30525
Columns: 19

Column names:
['customer_id', 'month', 'workouts_completed', 'minutes_active', 'classes_booked', 'support_tickets', 'minutes_active_missing_flag', 'signup_month', 'is_first_month', 'prev_month_workouts', 'prev_month_minutes', 'prev_month_classes', 'prev_month_support_tickets', 'prev_month_engagement_band', 'support_contact_flag', 'prev_month_support_flag', 'churn_flag', 'churn_month', 'churn_month_flag']
